# Common JSON Tools

Use this notebook to read a JSON file and display its content as tables for easier checking.

In [1]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import display

In [2]:
# Set the JSON file to inspect.
# You can replace this with another file path when needed.
json_file_path = Path("dcc/schema_register.json")

if not json_file_path.exists():
    fallback_candidates = [
        Path("schema_register.json"),
        Path("dcc/schema_register.json"),
    ]
    json_file_path = next((path for path in fallback_candidates if path.exists()), json_file_path)

print(f"JSON file path: {json_file_path}")

JSON file path: schema_register.json


In [3]:
with json_file_path.open("r", encoding="utf-8") as f:
    json_data = json.load(f)

print("Top-level keys:")
print(list(json_data.keys()) if isinstance(json_data, dict) else type(json_data))

Top-level keys:
['register_name', 'source', 'parameters', 'row_mapping', 'schema']


In [4]:
def to_schema_table(data):
    schema_dict = data.get("schema", {}) if isinstance(data, dict) else {}
    rows = [
        {"standard_name": standard_name, "source_name": source_name}
        for standard_name, source_names in schema_dict.items()
        for source_name in source_names
    ]
    return pd.DataFrame(rows)


def to_parameters_table(data):
    parameters_dict = data.get("parameters", {}) if isinstance(data, dict) else {}
    rows = [
        {"parameter": key, "value": value}
        for key, value in parameters_dict.items()
    ]
    return pd.DataFrame(rows)


def to_row_mapping_table(data):
    row_mapping_dict = data.get("row_mapping", {}) if isinstance(data, dict) else {}
    rows = [
        {"approval_code": approval_code, "alias": alias}
        for approval_code, aliases in row_mapping_dict.items()
        for alias in aliases
    ]
    return pd.DataFrame(rows)


def to_top_level_table(data):
    if not isinstance(data, dict):
        return pd.DataFrame([{"type": str(type(data)), "value": data}])

    rows = []
    for key, value in data.items():
        if isinstance(value, dict):
            value_type = "dict"
            size = len(value)
        elif isinstance(value, list):
            value_type = "list"
            size = len(value)
        else:
            value_type = type(value).__name__
            size = None

        rows.append({"key": key, "type": value_type, "size": size, "preview": str(value)[:120]})

    return pd.DataFrame(rows)

In [5]:
top_level_table = to_top_level_table(json_data)
schema_table = to_schema_table(json_data)
parameters_table = to_parameters_table(json_data)
row_mapping_table = to_row_mapping_table(json_data)

print("Top-Level Summary")
display(top_level_table)

print("Schema Table")
display(schema_table)

print("Parameters Table")
display(parameters_table)

print("Row Mapping Table")
display(row_mapping_table)

Top-Level Summary


,key,type,size,preview
0,register_name,str,NaN,dcc_schema_register
1,source,dict,3.0,"{'type': 'notebook', 'path': 'dcc/dcc_mdl.ipyn..."
2,parameters,dict,16.0,"{'start_col': 'P', 'end_col': 'AP', 'header_ro..."
3,row_mapping,dict,7.0,"{'REJ': ['Rejected', 'REJ'], 'NAP': ['Not Appr..."
4,schema,dict,46.0,{'Document_ID': ['CES_SALCON-SDC JV Cor Ref No...


Schema Table


,standard_name,source_name
0,Document_ID,CES_SALCON-SDC JV Cor Ref No
1,Document_Title,Document Description
2,Document_Revision,Rev
3,Document_Revision,This Revision
4,Department,Dept
5,Discipline,Discipline
6,Project_Code,Proj. Code
7,Facility_Code,Proj. Prefix
8,Document_Type,Doc Type
9,Document_Sequence_Number,Number


Parameters Table


,parameter,value
0,start_col,P
1,end_col,AP
2,header_row_index,4
3,pending_status,Awaiting S.O.'s response
4,duration_is_working_day,True
5,first_review_duration,20
6,second_review_duration,14
7,resubmission_duration,14
8,progress_stage,not_start
9,pc_name,CESL-22120


Row Mapping Table


,approval_code,alias
0,REJ,Rejected
1,REJ,REJ
2,NAP,Not Approved - Revise and resubmit
3,NAP,Not Approved
4,AWC,Approved with Comments
5,AWC,Approved as noted
6,APP,Approved
7,APP,APP
8,INF,For Information
9,INF,INF
